In [104]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [114]:
class Value:
    def __init__(self, data, _children = (), label = '', grad = 0.0, _op = ''):
        self.data = data
        self._prev = set(_children)
        self.label = label
        self.grad = grad
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f"Value {self.label} = {self.data}"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), _op = '+')

        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
        out._backward = _backward
        
        return out
    
    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other),  _op = '*')

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __neg__(self):
        return self * -1

    def tanh(self):
        t = (math.exp(2 * self.data) - 1) / (math.exp(2 * self.data) + 1)
        out = Value(t, (self, ), _op = 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.grad * out.data
        out._backward = _backward

        return out

    def backward(self):
        list = []
        set1 = set()
        topsort1 = self.topsort(list, set1)

        for i in range(len(topsort1)):
            topsort1[len(topsort1) - i - 1]._backward()

    def topsort(self, topsort = [], visited = set()):
        if self in visited:
            return
        
        visited.add(self)

        for child in self._prev:
            child.topsort(topsort, visited)
        topsort.append(self)
        return topsort

In [117]:
class Neuron:
    def __init__(self, nin):
        self.weights = [Value(random.uniform(-1.0, 1.0)) for _ in range(nin)]
        self.bias = Value(random.uniform(-1.0, 1.0))

    def __call__(self, inp):
        out = sum(wi * xi for wi, xi in zip(self.weights, inp)) + self.bias
        return out.tanh()

class Layer:
    def __init__(self, nin, nout):
        #neurons
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out


class MLP:
    def __init__(self, nin, nouts):
        #size and layers
        size = [nin] + nouts
        self.layers = [Layer(size[i], size[i + 1]) for i in range(len(size) - 1)]

    def __call__(self, x):
        out = x
        for layer in self.layers:
            print(len(layer.neurons))
            out = layer(out)
        return out


In [118]:
x = [1.0, 2.0, 3.0]

''' Neuron test
# n = Neuron(len(x))
# print(f"weights: {n.weights}, bias: {n.bias}")
# for wi, xi in zip(n.weights, x):
#     print(f"({wi} * {xi}) + ", end = "")
# print(f"{n.bias} = ")
# n(x)
'''


# layer = Layer(3, 4)
# layer(x)


mlp = MLP(3, [4, 4, 1])
mlp(x)

4
4
1


[Value  = 0.8807971321491067]